In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive/SST_Data"))

['GISS-E2-1-H_historical_1850_2014.nc', 'sst_anomaly_ensemble0.nc', 'COBE_SST_resize.nc', 'SeqVAE_Checkpoints', 'CMIP6_original_samples.nc', 'VAE_generated_samples.nc', 'Original_samples.nc']


In [3]:
import xarray as xr
nc_path = "/content/drive/MyDrive/SST_Data/GISS-E2-1-H_historical_1850_2014.nc"
ds = xr.open_dataset(nc_path)
print(ds)

<xarray.Dataset> Size: 3GB
Dimensions:   (ensemble: 25, years: 165, mon: 12, lat: 48, lon: 144)
Coordinates:
  * ensemble  (ensemble) <U41 4kB 'tos_Omon_GISS-E2-1-H_historical_r10i1p1f1'...
  * years     (years) int32 660B 1850 1851 1852 1853 ... 2011 2012 2013 2014
  * mon       (mon) int32 48B 0 1 2 3 4 5 6 7 8 9 10 11
  * lat       (lat) float64 384B -88.12 -84.38 -80.62 ... 80.62 84.38 88.12
  * lon       (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
Data variables:
    sst       (ensemble, years, mon, lat, lon) float64 3GB ...


In [4]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import detrend
from numpy.linalg import svd

nc_path = "/content/drive/MyDrive/SST_Data/GISS-E2-1-H_historical_1850_2014.nc"
ds = xr.open_dataset(nc_path)

sst = ds["sst"]

In [5]:
ensemble_id = 0

sst_ens = sst.isel(ensemble=ensemble_id)
sst_ens = sst_ens.fillna(0)
print(f"Using ensemble {ensemble_id}")

Using ensemble 0


In [6]:
import os
print(os.listdir("/content/drive/MyDrive/SST_Data"))

nc_path = "/content/drive/MyDrive/SST_Data/GISS-E2-1-H_historical_1850_2014.nc"

['GISS-E2-1-H_historical_1850_2014.nc', 'sst_anomaly_ensemble0.nc', 'COBE_SST_resize.nc', 'SeqVAE_Checkpoints', 'CMIP6_original_samples.nc', 'VAE_generated_samples.nc', 'Original_samples.nc']


In [7]:
nc_path = "/content/drive/MyDrive/SST_Data/GISS-E2-1-H_historical_1850_2014.nc"
ds = xr.open_dataset(nc_path)

ensemble_id = 0
sst = ds["sst"].isel(ensemble=ensemble_id)

In [8]:
import xarray as xr
import numpy as np
import torch
from scipy.signal import detrend
from torch.utils.data import Dataset, DataLoader


nc_path = "/content/drive/MyDrive/SST_Data/GISS-E2-1-H_historical_1850_2014.nc"
ds = xr.open_dataset(nc_path)

ensemble_id = 0
sst = ds["sst"].isel(ensemble=ensemble_id)

print("Original SST shape:", sst.shape)

Original SST shape: (165, 12, 48, 144)


In [9]:

# ==============================
# Train/Test split by years
# ==============================

train_years = slice(0,115)     # 1850–1965
test_years  = slice(115,None)  # 1966–2014

sst_train = sst.isel(years=train_years)
sst_test  = sst.isel(years=test_years)

print("Train years:", sst_train.shape)
print("Test years:", sst_test.shape)

Train years: (115, 12, 48, 144)
Test years: (50, 12, 48, 144)


In [10]:
# ==============================
# Computing climatology (TRAIN ONLY)
# ==============================

climatology = sst_train.mean(dim="years")

sst_train_anom = sst_train - climatology
sst_test_anom  = sst_test  - climatology

In [11]:

# ==============================
# Removing linear trend
# ==============================

sst_train_dt = xr.apply_ufunc(
    detrend,
    sst_train_anom.fillna(0),
    input_core_dims=[["years"]],
    output_core_dims=[["years"]],
    kwargs={"type": "linear"},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[sst_train_anom.dtype],
)

sst_test_dt = xr.apply_ufunc(
    detrend,
    sst_test_anom.fillna(0),
    input_core_dims=[["years"]],
    output_core_dims=[["years"]],
    kwargs={"type": "linear"},
    vectorize=True,
    dask="parallelized",
    output_dtypes=[sst_test_anom.dtype],
)

In [12]:

# ==============================
# Converting (years,mon,lat,lon) -> (time,lat,lon)
# ==============================

train_time = (
    sst_train_dt
    .stack(time=("years","mon"))
    .transpose("time","lat","lon")
)

test_time = (
    sst_test_dt
    .stack(time=("years","mon"))
    .transpose("time","lat","lon")
)

train_time = train_time.fillna(0.0)
test_time  = test_time.fillna(0.0)

print("Train time shape:", train_time.shape)
print("Test time shape:", test_time.shape)

Train time shape: (1380, 48, 144)
Test time shape: (600, 48, 144)


In [13]:
train_mean = train_time.mean()
train_std  = train_time.std()

train_norm = (train_time - train_mean) / train_std
test_norm  = (test_time  - train_mean) / train_std

In [14]:
train_data = train_norm.values
test_data_full = test_norm.values

print("Train samples:", train_data.shape)
print("Full test samples:", test_data_full.shape)

Train samples: (1380, 48, 144)
Full test samples: (600, 48, 144)


In [15]:
# ==============================
# Validation/Test split (300 / 300)
# ==============================

val_data  = test_data_full[:300]
test_data = test_data_full[300:600]

print("Train:", train_data.shape)
print("Val:", val_data.shape)
print("Test:", test_data.shape)


Train: (1380, 48, 144)
Val: (300, 48, 144)
Test: (300, 48, 144)


In [16]:
class SSTVAEDataset(Dataset):

    def __init__(self,data):
        self.data = torch.tensor(data,dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self,idx):
        x = self.data[idx]
        return x,x


train_dataset = SSTVAEDataset(train_data)
val_dataset   = SSTVAEDataset(val_data)
test_dataset  = SSTVAEDataset(test_data)

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Dataloaders created successfully.")

Dataloaders created successfully.
